# 03. Results Analysis

Reads `results/metrics.csv` and `results/clip_scores.csv`, produces the plots and tables for the report. Run after benchmarks complete.

Sections:
1. Per-config latency (mean +/- std)
2. Speedup vs FP32 GPU baseline
3. Quality (CLIP score) vs config
4. Latency breakdown by prompt (sanity check for outliers)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

metrics = pd.read_csv('../results/metrics.csv')
print(f'Loaded {len(metrics)} rows across {metrics.run_id.nunique()} runs')
metrics.head()

In [ ]:
# Per-config latency summary
summary = metrics.groupby('run_id').wall_time_s.agg(['mean', 'std', 'min', 'max', 'count'])
summary = summary.round(3).sort_index()
summary

In [ ]:
# Speedup vs GPU FP32 baseline
baseline = summary.loc['02_gpu_baseline_fp32', 'mean']
speedup = summary[['mean']].copy()
speedup['speedup_vs_fp32_baseline'] = baseline / speedup['mean']
speedup = speedup.round(2)
speedup

In [ ]:
# Latency bar chart
fig, ax = plt.subplots(figsize=(10, 5))
summary['mean'].plot(kind='bar', yerr=summary['std'], ax=ax, capsize=4)
ax.set_ylabel('Wall time per image (s)')
ax.set_xlabel('Configuration')
ax.set_title('Single-image latency by configuration (lower is better)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../results/fig_latency_by_config.png', dpi=150)
plt.show()

In [ ]:
# CLIP score - quality regression check
clip_path = Path('../results/clip_scores.csv')
if clip_path.exists():
    clip = pd.read_csv(clip_path)
    clip_summary = clip.groupby('run_id').clip_score.agg(['mean', 'std']).round(4)
    print(clip_summary)
else:
    print('No CLIP scores yet - run `python -m src.quality` after benchmarks')

In [ ]:
# Sanity check - any outlier prompts skewing the mean?
by_prompt = metrics.groupby(['run_id', 'prompt_id']).wall_time_s.mean().unstack()
by_prompt.round(2)